# NQ Initial Balance — Order Flow Analysis (2026)
Tick data from `NQ_ticks_2026.parquet`  
Schema: `timestamp | price | volume | aggressor ('Buy'/'Sell')`

**What's new vs your original IB notebook:**
- All analysis sliced by **delta bias** (buy-dominated vs sell-dominated IB)
- **Delta divergence** flag (price direction ≠ delta direction inside IB)
- **Cumulative delta shape** within IB (trending vs reversing)
- All your existing metrics (breaks, MFE/MAE, midpoint retest) preserved and cross-tabbed with delta
- Works directly off tick data — no pre-resampling needed

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────────────────
TICK_FILE = r'NQ_ticks_2026.parquet'   # adjust path as needed

IB_START_TIME  = '09:30'
IB_END_TIME    = '10:30'
NY_END_TIME    = '16:00'
POST_IB_WINDOW = '11:30'   # 1h after IB ends

# ── LOAD TICKS ────────────────────────────────────────────────────────────
ticks = pd.read_parquet(TICK_FILE)

# Normalise timestamp → US/Eastern
if 'timestamp' in ticks.columns:
    ticks['timestamp'] = pd.to_datetime(ticks['timestamp'], utc=True)
    ticks = ticks.set_index('timestamp')
else:
    ticks.index = pd.to_datetime(ticks.index, utc=True)

ticks = ticks.sort_index()
ticks.index = ticks.index.tz_convert('US/Eastern')

# Normalise aggressor labels
ticks['aggressor'] = ticks['aggressor'].str.strip().str.capitalize()   # 'Buy' | 'Sell'

# Per-tick delta
ticks['buy_vol']  = np.where(ticks['aggressor'] == 'Buy',  ticks['volume'], 0)
ticks['sell_vol'] = np.where(ticks['aggressor'] == 'Sell', ticks['volume'], 0)
ticks['delta']    = ticks['buy_vol'] - ticks['sell_vol']

# NY trading day label (18:00 anchor, same as your notebooks)
ticks['NY_Trading_Day'] = (ticks.index - pd.Timedelta(hours=18)).normalize()

print(f'Loaded {len(ticks):,} ticks')
print(f'Range : {ticks.index.min()} → {ticks.index.max()}')
print(f'Columns: {ticks.columns.tolist()}')
ticks.head(3)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# RESAMPLE TO 1-MIN BARS  (used for OHLC lookups)
# ─────────────────────────────────────────────────────────────────────────
bars = ticks.resample('1min').agg(
    open=('price', 'first'),
    high=('price', 'max'),
    low=('price', 'min'),
    close=('price', 'last'),
    volume=('volume', 'sum'),
    buy_vol=('buy_vol', 'sum'),
    sell_vol=('sell_vol', 'sum'),
    delta=('delta', 'sum'),
    trades=('price', 'count'),
).dropna(subset=['open'])

bars['cum_delta'] = bars['delta'].cumsum()
bars['NY_Trading_Day'] = (bars.index - pd.Timedelta(hours=18)).normalize()

print(f'{len(bars):,} 1-min bars')
bars.tail(3)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# BUILD IB RECORD PER DAY
# ─────────────────────────────────────────────────────────────────────────
trading_days = sorted(bars['NY_Trading_Day'].unique())
ib_records = []

for day in trading_days[1:]:   # skip first (may be partial)

    # Slice IB bars
    day_bars = bars[bars['NY_Trading_Day'] == day]
    t = day_bars.index.time

    ib_bars  = day_bars[(t >= pd.Timestamp(IB_START_TIME).time()) &
                        (t <  pd.Timestamp(IB_END_TIME).time())]
    post_ib  = day_bars[(t >= pd.Timestamp(IB_END_TIME).time()) &
                        (t <  pd.Timestamp(NY_END_TIME).time())]
    post_1h  = day_bars[(t >= pd.Timestamp(IB_END_TIME).time()) &
                        (t <  pd.Timestamp(POST_IB_WINDOW).time())]

    if ib_bars.empty or post_ib.empty:
        continue

    # ── OHLCV basics ──────────────────────────────────────────────────────
    ib_open  = ib_bars['open'].iloc[0]
    ib_close = ib_bars['close'].iloc[-1]
    ib_high  = ib_bars['high'].max()
    ib_low   = ib_bars['low'].min()
    ib_range = ib_high - ib_low
    ib_mid   = (ib_high + ib_low) / 2

    if pd.isna([ib_open, ib_close, ib_high, ib_low]).any() or ib_range == 0:
        continue

    ib_direction   = ('bullish' if ib_close > ib_open else
                      'bearish' if ib_close < ib_open else None)
    if ib_direction is None:
        continue

    ib_close_pos = 'above_mid' if ib_close > ib_mid else 'below_mid'

    high_time  = ib_bars.loc[ib_bars['high'] == ib_high].index[0]
    low_time   = ib_bars.loc[ib_bars['low']  == ib_low ].index[0]
    formed_first = 'high' if high_time < low_time else 'low'

    ib_range_pct = (ib_range / ib_open) * 100
    day_of_week  = high_time.strftime('%A')

    ny_close      = post_ib['close'].iloc[-1]
    ny_direction  = ('bullish' if ny_close > ib_open else
                     'bearish' if ny_close < ib_open else 'flat')

    # ── ORDER FLOW inside IB ──────────────────────────────────────────────
    ib_buy_vol   = ib_bars['buy_vol'].sum()
    ib_sell_vol  = ib_bars['sell_vol'].sum()
    ib_volume    = ib_bars['volume'].sum()
    ib_delta     = ib_bars['delta'].sum()          # net = buy - sell
    ib_delta_pct = ib_delta / ib_volume if ib_volume else np.nan   # -1..+1
    ib_delta_bias = 'buy' if ib_delta > 0 else 'sell'

    # Divergence: price dir ≠ delta dir
    ib_delta_divergence = (
        (ib_direction == 'bullish' and ib_delta_bias == 'sell') or
        (ib_direction == 'bearish' and ib_delta_bias == 'buy')
    )

    # Cumulative delta SHAPE within IB:
    # 'trending'  = cum delta moves consistently in one direction
    # 'reversing' = cum delta changes sign at least once
    ib_cum_delta = ib_bars['delta'].cumsum()
    cd_min, cd_max = ib_cum_delta.min(), ib_cum_delta.max()
    cd_final = ib_cum_delta.iloc[-1]
    # If final value is same sign as the max excursion → trending
    ib_delta_shape = (
        'trending'  if (cd_final > 0 and cd_min >= 0) or
                       (cd_final < 0 and cd_max <= 0)
        else 'reversing'
    )

    # ── BREAK ANALYSIS (same logic as original notebook) ─────────────────
    def find_first_break(data, ib_h, ib_l):
        h_broken, l_broken = False, False
        h_time,   l_time   = None,  None
        for idx, row in data.iterrows():
            if not h_broken and row['high'] > ib_h:
                h_broken, h_time = True, idx
            if not l_broken and row['low']  < ib_l:
                l_broken, l_time = True, idx
        return h_broken, h_time, l_broken, l_time

    ib_end_ts = ib_bars.index[-1]   # last bar timestamp inside IB

    hb_1h, hb_t_1h, lb_1h, lb_t_1h = find_first_break(post_1h,  ib_high, ib_low)
    hb_ses, hb_t_ses, lb_ses, lb_t_ses = find_first_break(post_ib, ib_high, ib_low)

    def mins_after(t):
        if t is None: return None
        return (t - ib_end_ts).total_seconds() / 60

    # ── MFE / MAE (same as original) ──────────────────────────────────────
    def mfe_mae(data, break_time, direction):
        """direction: 'high' (break upward) or 'low' (break downward)"""
        if break_time is None or data.empty:
            return None, None
        after = data.loc[break_time:]
        if after.empty:
            return None, None
        if direction == 'high':
            ref = ib_high
            mfe = (after['high'].max() - ref) / ref * 100
            mae_pts = ref - after['low'].min()
            mae = max(mae_pts, 0) / ref * 100
        else:
            ref = ib_low
            mfe = (ref - after['low'].min()) / ref * 100
            mae_pts = after['high'].max() - ref
            mae = max(mae_pts, 0) / ref * 100
        return mfe, mae

    hmfe_1h,  hmae_1h  = mfe_mae(post_1h,  hb_t_1h,  'high')
    hmfe_ses, hmae_ses = mfe_mae(post_ib,  hb_t_ses, 'high')
    lmfe_1h,  lmae_1h  = mfe_mae(post_1h,  lb_t_1h,  'low')
    lmfe_ses, lmae_ses = mfe_mae(post_ib,  lb_t_ses, 'low')

    # ── MIDPOINT RETEST ───────────────────────────────────────────────────
    threshold = ib_mid * 0.001
    left_mid, left_mid_t, mid_retested, mid_retest_mins = False, None, False, None
    for idx, row in post_ib.iterrows():
        if not left_mid:
            if abs(row['close'] - ib_mid) > threshold:
                left_mid, left_mid_t = True, idx
        else:
            if abs(row['low'] - ib_mid) <= threshold or abs(row['high'] - ib_mid) <= threshold:
                mid_retested = True
                mid_retest_mins = (idx - left_mid_t).total_seconds() / 60
                break

    # ── STORE ─────────────────────────────────────────────────────────────
    ib_records.append({
        # Identity
        'date':               day,
        'day_of_week':        day_of_week,

        # IB price structure
        'ib_open':            ib_open,
        'ib_close':           ib_close,
        'ib_high':            ib_high,
        'ib_low':             ib_low,
        'ib_range':           ib_range,
        'ib_range_pct':       ib_range_pct,
        'ib_mid':             ib_mid,
        'ib_direction':       ib_direction,
        'ib_close_pos':       ib_close_pos,
        'formed_first':       formed_first,

        # Order flow — IB
        'ib_volume':          ib_volume,
        'ib_buy_vol':         ib_buy_vol,
        'ib_sell_vol':        ib_sell_vol,
        'ib_delta':           ib_delta,
        'ib_delta_pct':       ib_delta_pct,
        'ib_delta_bias':      ib_delta_bias,
        'ib_delta_divergence':ib_delta_divergence,
        'ib_delta_shape':     ib_delta_shape,

        # Breaks — 1h
        'ib_high_broken_1h':  hb_1h,
        'ib_low_broken_1h':   lb_1h,
        'time_to_high_break_mins': mins_after(hb_t_1h),
        'time_to_low_break_mins':  mins_after(lb_t_1h),

        # Breaks — session
        'ib_high_broken_session': hb_ses,
        'ib_low_broken_session':  lb_ses,

        # MFE / MAE
        'high_mfe_1h_pct':    hmfe_1h,
        'high_mae_1h_pct':    hmae_1h,
        'high_mfe_session_pct': hmfe_ses,
        'high_mae_session_pct': hmae_ses,
        'low_mfe_1h_pct':     lmfe_1h,
        'low_mae_1h_pct':     lmae_1h,
        'low_mfe_session_pct': lmfe_ses,
        'low_mae_session_pct': lmae_ses,

        # Midpoint
        'mid_retested':       mid_retested,
        'mid_retest_mins':    mid_retest_mins,

        # Outcome
        'ny_direction':       ny_direction,
    })

ib = pd.DataFrame(ib_records)

# Z-score bands (same as original)
ib['ib_range_zscore'] = stats.zscore(ib['ib_range_pct'])
def zscore_band(z):
    if z < -1.5: return 'Very Small'
    if z < -0.5: return 'Small'
    if z <  0.5: return 'Average'
    if z <  1.5: return 'Large'
    return 'Very Large'
ib['range_band'] = ib['ib_range_zscore'].apply(zscore_band)

print(f'IB records built: {len(ib)}')
print(ib[['date','ib_direction','ib_delta_bias','ib_delta_divergence','ib_delta_shape']].head(10).to_string())

## 1 · Overview — Delta Bias & Divergence

In [ ]:
n = len(ib)
print(f'Total IB days: {n}\n')

# --- Delta bias distribution ---
print('── Delta Bias ──────────────────────────────')
for bias in ['buy', 'sell']:
    c = (ib['ib_delta_bias'] == bias).sum()
    print(f'  {bias.capitalize()}-dominated IB : {c:3d}  ({c/n*100:.1f}%)')

print()

# --- Direction × Delta cross-tab ---
print('── IB Price Direction × Delta Bias ─────────')
ct = pd.crosstab(ib['ib_direction'], ib['ib_delta_bias'], margins=True)
print(ct.to_string())

print()

# --- Divergence ---
div = ib['ib_delta_divergence'].sum()
print(f'── Divergence (price dir ≠ delta dir) ──────')
print(f'  {div:3d} / {n}  ({div/n*100:.1f}%)')

print()

# --- Delta shape ---
print('── Cum-Delta Shape within IB ───────────────')
for shape in ['trending', 'reversing']:
    c = (ib['ib_delta_shape'] == shape).sum()
    print(f'  {shape.capitalize():10s}: {c:3d}  ({c/n*100:.1f}%)')

## 2 · IB Break Rate — sliced by Delta Bias

In [ ]:
def break_stats(df, label):
    n = len(df)
    if n == 0:
        print(f'  {label}: no data'); return
    hb  = df['ib_high_broken_1h'].sum()
    lb  = df['ib_low_broken_1h'].sum()
    both = (df['ib_high_broken_1h'] & df['ib_low_broken_1h']).sum()
    neither = (~df['ib_high_broken_1h'] & ~df['ib_low_broken_1h']).sum()
    hb_s  = df['ib_high_broken_session'].sum()
    lb_s  = df['ib_low_broken_session'].sum()
    print(f'  {label}  (n={n})')
    print(f'    High broken  1h  : {hb:3d} ({hb/n*100:5.1f}%)')
    print(f'    Low  broken  1h  : {lb:3d} ({lb/n*100:5.1f}%)')
    print(f'    Both broken  1h  : {both:3d} ({both/n*100:5.1f}%)')
    print(f'    Neither      1h  : {neither:3d} ({neither/n*100:5.1f}%)')
    print(f'    High broken  ses : {hb_s:3d} ({hb_s/n*100:5.1f}%)')
    print(f'    Low  broken  ses : {lb_s:3d} ({lb_s/n*100:5.1f}%)')
    print()

print('═'*55)
print('IB BREAK RATES BY DELTA BIAS')
print('═'*55)
for bias in ['buy', 'sell']:
    break_stats(ib[ib['ib_delta_bias'] == bias], f'Delta = {bias.upper()}')

print('─'*55)
print('FURTHER SPLIT: Direction × Delta Bias')
print('─'*55)
for direction in ['bullish', 'bearish']:
    for bias in ['buy', 'sell']:
        subset = ib[(ib['ib_direction'] == direction) & (ib['ib_delta_bias'] == bias)]
        break_stats(subset, f'{direction.capitalize()} IB + {bias.upper()} delta')

## 3 · Divergence Days — What Happens Next?

In [ ]:
div_days  = ib[ib['ib_delta_divergence'] == True]
aln_days  = ib[ib['ib_delta_divergence'] == False]

def divergence_summary(df, label):
    n = len(df)
    if n == 0: return
    print(f'\n{label}  (n={n})')

    # High/low break rate
    hb = df['ib_high_broken_1h'].sum()
    lb = df['ib_low_broken_1h'].sum()
    print(f'  High broken 1h : {hb:3d} ({hb/n*100:5.1f}%)')
    print(f'  Low  broken 1h : {lb:3d} ({lb/n*100:5.1f}%)')

    # NY session alignment with IB price direction
    for ib_dir in ['bullish', 'bearish']:
        sub = df[df['ib_direction'] == ib_dir]
        if sub.empty: continue
        aligned = (sub['ny_direction'] == ib_dir).sum()
        print(f'  {ib_dir.capitalize()} IB → NY also {ib_dir}: {aligned}/{len(sub)} ({aligned/len(sub)*100:.1f}%)')

    # Mean delta pct
    print(f'  Mean |delta_pct| : {df["ib_delta_pct"].abs().mean():.3f}')

divergence_summary(div_days,  '▶ DIVERGENCE days (price dir ≠ delta)')
divergence_summary(aln_days,  '▶ ALIGNED days   (price dir = delta)')

## 4 · MFE / MAE after IB Breaks — by Delta Bias

In [ ]:
def mfe_mae_table(df, label):
    n = len(df)
    if n == 0: return
    print(f'\n{label}  (n={n})')

    for side, mfe_col, mae_col in [
        ('HIGH break 1h',      'high_mfe_1h_pct',      'high_mae_1h_pct'),
        ('HIGH break session',  'high_mfe_session_pct', 'high_mae_session_pct'),
        ('LOW  break 1h',       'low_mfe_1h_pct',       'low_mae_1h_pct'),
        ('LOW  break session',  'low_mfe_session_pct',  'low_mae_session_pct'),
    ]:
        mfe = df[mfe_col].dropna()
        mae = df[mae_col].dropna()
        if mfe.empty: continue
        print(f'  {side}')
        print(f'    MFE  mean={mfe.mean():.3f}%  median={mfe.median():.3f}%  n={len(mfe)}')
        print(f'    MAE  mean={mae.mean():.3f}%  median={mae.median():.3f}%')

print('═'*60)
print('MFE / MAE BY DELTA BIAS')
print('═'*60)
for bias in ['buy', 'sell']:
    mfe_mae_table(ib[ib['ib_delta_bias'] == bias], f'Delta = {bias.upper()}')

print('\n' + '─'*60)
print('DIVERGENCE vs ALIGNED')
print('─'*60)
mfe_mae_table(div_days, 'DIVERGENCE days')
mfe_mae_table(aln_days, 'ALIGNED days')

## 5 · Delta Shape (Trending vs Reversing) × Breaks

In [ ]:
print('═'*60)
print('CUM-DELTA SHAPE WITHIN IB × BREAK OUTCOMES')
print('═'*60)
for shape in ['trending', 'reversing']:
    break_stats(ib[ib['ib_delta_shape'] == shape], f'Delta shape = {shape.upper()}')

print('─'*60)
print('SHAPE × DIRECTION')
print('─'*60)
for shape in ['trending', 'reversing']:
    for direction in ['bullish', 'bearish']:
        subset = ib[(ib['ib_delta_shape'] == shape) & (ib['ib_direction'] == direction)]
        break_stats(subset, f'{shape.capitalize()} delta + {direction.capitalize()} IB')

## 6 · Midpoint Retest — by Delta Bias

In [ ]:
print('═'*55)
print('MIDPOINT RETEST BY DELTA BIAS')
print('═'*55)
for bias in ['buy', 'sell']:
    sub = ib[ib['ib_delta_bias'] == bias]
    n = len(sub)
    r = sub['mid_retested'].sum()
    times = sub['mid_retest_mins'].dropna()
    print(f'  {bias.upper()}-dominated  (n={n})')
    print(f'    Retested     : {r:3d} ({r/n*100:.1f}%)')
    if not times.empty:
        print(f'    Time to retest  mean={times.mean():.0f}m  median={times.median():.0f}m')
    print()

## 7 · IB Range Size × Delta — does a large IB have a different delta profile?

In [ ]:
print('═'*60)
print('RANGE BAND × DELTA METRICS')
print('═'*60)
band_order = ['Very Small', 'Small', 'Average', 'Large', 'Very Large']
summary = (
    ib.groupby('range_band')
    .agg(
        n=('ib_delta', 'count'),
        mean_range=('ib_range', 'mean'),
        buy_dom_pct=('ib_delta_bias', lambda x: (x=='buy').mean()*100),
        mean_delta_pct=('ib_delta_pct', 'mean'),
        divergence_pct=('ib_delta_divergence', lambda x: x.mean()*100),
        high_break_1h=('ib_high_broken_1h', lambda x: x.mean()*100),
        low_break_1h=('ib_low_broken_1h',  lambda x: x.mean()*100),
    )
    .reindex([b for b in band_order if b in ib['range_band'].values])
    .round(1)
)
print(summary.to_string())

## 8 · Day-of-Week × Delta

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday']
dow_summary = (
    ib.groupby('day_of_week')
    .agg(
        n=('ib_delta', 'count'),
        buy_dom_pct=('ib_delta_bias', lambda x: (x=='buy').mean()*100),
        mean_delta_pct=('ib_delta_pct', 'mean'),
        divergence_pct=('ib_delta_divergence', lambda x: x.mean()*100),
        high_break_1h=('ib_high_broken_1h', lambda x: x.mean()*100),
        low_break_1h=('ib_low_broken_1h',  lambda x: x.mean()*100),
    )
    .reindex([d for d in dow_order if d in ib['day_of_week'].values])
    .round(1)
)
print(dow_summary.to_string())

## 9 · Export

In [ ]:
out = 'NQ_IB_orderflow_2026.csv'
ib.to_csv(out, index=False)
print(f'Saved {len(ib)} rows → {out}')
print('\nColumns:')
print(ib.dtypes.to_string())